In [43]:
import json

with open("outputs/med_qa_fields.jsonl", "r") as f:
    examples = [json.loads(line) for line in f]

with open("outputs/pubmed_qa_fields.jsonl", "r") as f:
    examples.extend([json.loads(line) for line in f])

In [44]:
examples[0]

{'id': 0,
 'dataset': 'medqa',
 'prompt': 'Classify the medical field(s) for this QA example.\n\nQuestion: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?',
 'topics': ['orthopedics', 'surgery']}

In [45]:
keep_topics = ["infectious_disease", "nephrology", "emergency_medicine"]

kept_examples = []
for ex in examples:
    if (set(ex['topics']) & set(keep_topics)) != set():  # at least one overlap
        kept_examples.append(ex)

In [46]:
len(kept_examples)

710

In [47]:
tot_counts = [0] * 3
overlap_counts = [0] * 3
for ex in kept_examples:
    overlap = set(ex['topics']) & set(keep_topics)
    if len(overlap) >= 2:
        overlap_counts[0] += int(keep_topics[0] in overlap)
        overlap_counts[1] += int(keep_topics[1] in overlap)
        overlap_counts[2] += int(keep_topics[2] in overlap)
    tot_counts[0] += int(keep_topics[0] in ex['topics'])
    tot_counts[1] += int(keep_topics[1] in ex['topics'])
    tot_counts[2] += int(keep_topics[2] in ex['topics'])
print(f"{keep_topics[0]}".ljust(20) + f"| {overlap_counts[0] / tot_counts[0]:.2f}")
print(f"{keep_topics[1]}".ljust(20) + f"| {overlap_counts[1] / tot_counts[1]:.2f}")
print(f"{keep_topics[2]}".ljust(20) + f"| {overlap_counts[2] / tot_counts[2]:.2f}")

infectious_disease  | 0.18
nephrology          | 0.15
emergency_medicine  | 0.16


In [48]:
kept_examples[0]

{'id': 2,
 'dataset': 'medqa',
 'prompt': 'Classify the medical field(s) for this QA example.\n\nQuestion: Two weeks after undergoing an emergency cardiac catherization with stenting for unstable angina pectoris, a 61-year-old man has decreased urinary output and malaise. He has type 2 diabetes mellitus and osteoarthritis of the hips. Prior to admission, his medications were insulin and naproxen. He was also started on aspirin, clopidogrel, and metoprolol after the coronary intervention. His temperature is 38°C (100.4°F), pulse is 93/min, and blood pressure is 125/85 mm Hg. Examination shows mottled, reticulated purplish discoloration of the feet. Laboratory studies show:\nHemoglobin count 14 g/dL\nLeukocyte count 16,400/mm3\nSegmented neutrophils 56%\nEosinophils 11%\nLymphocytes 31%\nMonocytes 2%\nPlatelet count 260,000/mm3\nErythrocyte sedimentation rate 68 mm/h\nSerum\nUrea nitrogen 25 mg/dL\nCreatinine 4.2 mg/dL\nRenal biopsy shows intravascular spindle-shaped vacuoles. Which of t

In [50]:
from datasets import load_dataset

medqa = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

In [ ]:
# question

In [57]:
medqa[0].keys()

dict_keys(['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'])

In [56]:
pubmedqa[0].keys()

dict_keys(['pubid', 'question', 'context', 'long_answer', 'final_decision'])

In [59]:
pubmedqa[0]

{'pubid': 21645374,
 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
   'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), ce

In [65]:
from typing import Any, Dict
from functools import partial

def build_prompt(example: Dict[str, Any], dataset_name: str) -> str:
    if example.get("prompt"):
        return str(example.get("prompt"))
    if dataset_name == "pubmed_qa":
        question = example.get("question") or ""
        context = example.get("context") or example.get("abstract") or ""
    else:
        question = example.get("question") or example.get("query") or ""
        context = example.get("context") or example.get("passage") or ""
    options = ""
    choices = example.get("choices") or example.get("options")
    if choices:
        if isinstance(choices, list):
            labels = [chr(ord("A") + idx) for idx in range(len(choices))]
            texts = [str(item) for item in choices]
        else:
            labels = choices.get("label") or choices.get("labels")
            texts = choices.get("text") or choices.get("texts") or choices.get("choices")
        if labels and texts:
            paired = [f"{lbl}. {txt}" for lbl, txt in zip(labels, texts)]
            options = "\n" + "\n".join(paired)
    if dataset_name == "pubmed_qa" and context:
        return (
            "You are a medical QA assistant. Answer the question based on the context."
            f"\n\nContext:\n{context}\n\nQuestion: {question}{options}\nAnswer:"
        )
    return (
        "You are a medical QA assistant. Answer the question."
        f"\n\nQuestion: {question}{options}\nAnswer:"
    )


def extract_gold_answer(example: Dict[str, Any]) -> Any:
    topics = example.get("topics")
    if isinstance(topics, list) and topics:
        return topics
    if "final_decision" in example:
        return str(example.get("final_decision") or "")
    for key in (
        "answer",
        "final_answer",
        "exact_answer",
        "ideal_answer",
        "answer_text",
        "label",
    ):
        value = example.get(key)
        if value:
            if isinstance(value, list):
                return str(value[0])
            return str(value)
    choice_answer = extract_choice_answer(example)
    if choice_answer:
        return choice_answer
    return ""


map_fn = lambda name, ex: {
        "prompt": build_prompt(ex, name),
        "gold": extract_gold_answer(ex),
}

medqa_processed = medqa.map(partial(map_fn, "med_qa"), num_proc=4)
pubmedqa_processed = pubmedqa.map(partial(map_fn, "pubmed_qa"), num_proc=4)

Map (num_proc=4): 100%|████████████████████████████████████████████| 1000/1000 [00:00<00:00, 7629.05 examples/s]


In [67]:
pubmedqa_processed[0]

{'pubid': 21645374,
 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
   'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), ce

In [39]:
import json

with open("med_pubmed_top3_eval.jsonl", "w") as f:
    for example in kept_examples:
        f.write(json.dumps(example) + '\n')

In [42]:
with open("med_pubmed_top3_eval.jsonl", "r") as f:
    print(len([json.loads(line) for line in f]))

710


In [69]:
medqa[0]

{'question': 'A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?',
 'answer': 'Tell the attending that he cannot fail to disclose this mistake',
 'options': {'A': 'Disclose the error to the patient and put it in the operative report',
  'B': 'Tell the attending that he cannot fail to disclose this mistake',
  'C': 'Report the physician to the ethics committee',
  'D': 'Refuse to dictate the operative report'},
 'meta_info': 'step1',
 'ans

In [70]:
pubmedqa[0]

{'pubid': 21645374,
 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
   'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), ce

In [74]:
set([ex.get("choices") for ex in pubmedqa])

{None}